In [1]:
import os, torch
import pandas as pd
import numpy as np
# from pathlib import Path
from pathlib import PureWindowsPath
from monai.transforms import (Compose, LoadImage, Resized, Spacingd, ScaleIntensityRanged)

# from sklearn.model_selection import train_test_split
import torch.nn.functional as F
from monai.data import MetaTensor

import sys
sys.path.append("/projects/net_contrast_classification/contrast_phase")

from Radiomics.radiomics_pipeline import multi_channel
from Preprocessing.Contrast_data.contrast_preprocess import group_stratified_train_val_test_split
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
from sklearn.model_selection import GroupShuffleSplit
from itertools import product


In [ ]:
def create_paired_data(dataset):
    dataset = dataset.copy()

    # Assign server folder
    dataset["server_folder"] = dataset["exist_on_server"].apply(
        lambda x: "/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTINET"
        if pd.notna(x)
        else "/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTINET/not_on_server"
    )

    # Filter valid rows
    dataset = dataset[
        (dataset.contrast.isin(["Arterial", "Portal"])) &
        (dataset.is_liver_imaged.isin(["Yes", "Partially"])) &
        (dataset.phase_timing != "0.0")
    ].copy()

    # Convert time once
    dataset["AcquisitionTime_sec"] = pd.to_timedelta(
        dataset["AcquisitionTime"]
    ).dt.total_seconds()

    # Keep only exams with both phases
    required_phases = {"Arterial", "Portal"}

    paired_data = (
        dataset
        .sort_values(["SubjectKeyRadiology", "ExamDate", "AcquisitionTime_sec"])
        .groupby(["SubjectKeyRadiology", "ExamDate", "server_folder"])
        .filter(lambda g: required_phases.issubset(set(g["contrast"])))
    )
    
    return paired_data


def build_pairs(paired_data):

    pairs = []

    # Per exam
    for (patient_id, exam_date, server_folder), group in paired_data.groupby(
        ["SubjectKeyRadiology", "ExamDate", "server_folder"]
    ):

        group = group.sort_values("AcquisitionTime_sec")

        arterial = group[group["contrast"] == "Arterial"]
        portal = group[group["contrast"] == "Portal"]

        if arterial.empty or portal.empty:
            continue

        # For each arterial, pair with ALL future portal variants
        for _, a_row in arterial.iterrows():

            a_time = a_row["AcquisitionTime_sec"]

            future_portals = portal[portal["AcquisitionTime_sec"] > a_time]

            if future_portals.empty:
                continue

            a_file = os.path.join(
                server_folder,
                PureWindowsPath(a_row["MatchKey"]).name
            )

            # Preserve multiple kernel reconstructions
            for _, p_row in future_portals.iterrows():

                p_time = p_row["AcquisitionTime_sec"]
                interval = p_time - a_time

                if interval <= 0:
                    continue

                p_file = os.path.join(
                    server_folder,
                    PureWindowsPath(p_row["MatchKey"]).name
                )

                pairs.append({
                    "SubjectKeyRadiology": patient_id,
                    "ExamDate": exam_date,

                    "arterial_file": a_file,
                    "portal_file": p_file,

                    "arterial_organs": a_file.replace(".nii.gz", ".organs.nii.gz"),
                    "portal_organs": p_file.replace(".nii.gz", ".organs.nii.gz"),

                    # SAME arterial anchor
                    "time_interval": interval
                })

    return pd.DataFrame(pairs).reset_index(drop=True)


def files_load(data_dir, sample=None):

    dataset = pd.read_csv(data_dir)

    paired_data = create_paired_data(dataset)
    
    # -------- build pairs --------
    pairs_df = build_pairs(paired_data)

    # -------- optional sampling --------
    if sample:
        pairs_df = pairs_df.sample(n=sample, random_state=42)

    return pairs_df

In [11]:
# -------- Load paired data --------
data_dir = "/projects/net_contrast_classification/contrast_phase/data/cleaned_data_1.csv"
dataset = files_load(data_dir)
dataset

,SubjectKeyRadiology,ExamDate,arterial_file,portal_file,arterial_organs,portal_organs,time_interval
0,NKI-d23231-00-0002,2015-12-01,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,0.00519
1,NKI-d23231-00-0005,2010-02-04,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,53.64779
2,NKI-d23231-00-0005,2012-06-18,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,33.40000
3,NKI-d23231-00-0005,2022-05-11,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,30.84280
4,NKI-d23231-00-0005,2022-10-17,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,32.75200
...,...,...,...,...,...,...,...
1674,NKI-d23231-00-0887,2023-05-24,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,31.57800
1675,NKI-d23231-00-0887,2023-08-30,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,28.90000
1676,NKI-d23231-00-0888,2022-04-20,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,31.78189
1677,NKI-d23231-00-0888,2023-04-24,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,34.57700


In [26]:
train_idx, val_idx, test_idx = group_stratified_train_val_test_split(
                                                                    np.arange(len(dataset)),
                                                                    labels=pd.to_datetime(dataset["ExamDate"]).dt.year,
                                                                    groups=dataset["SubjectKeyRadiology"]
                                                                        )
dataset = dataset.reset_index(drop=True)


In [27]:
dataset.loc[train_idx, "split"] = "train"
dataset.loc[val_idx, "split"] = "val"
dataset.loc[test_idx, "split"] = "test"
dataset

,SubjectKeyRadiology,ExamDate,arterial_file,portal_file,arterial_organs,portal_organs,time_interval,split
0,NKI-d23231-00-0002,2015-12-01,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,0.00519,test
1,NKI-d23231-00-0005,2010-02-04,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,53.64779,train
2,NKI-d23231-00-0005,2012-06-18,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,33.40000,train
3,NKI-d23231-00-0005,2022-05-11,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,30.84280,train
4,NKI-d23231-00-0005,2022-10-17,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,32.75200,train
...,...,...,...,...,...,...,...,...
1674,NKI-d23231-00-0887,2023-05-24,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,31.57800,val
1675,NKI-d23231-00-0887,2023-08-30,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,28.90000,val
1676,NKI-d23231-00-0888,2022-04-20,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,31.78189,train
1677,NKI-d23231-00-0888,2023-04-24,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,34.57700,train


In [28]:
dataset.split.value_counts()


split
train    1006
test      337
val       336
Name: count, dtype: int64

In [29]:
print(dataset[dataset.split == "train"]["ExamDate"].value_counts())
print("\n")
print(dataset[dataset.split == "test"]["ExamDate"].value_counts())
print("\n")
print(dataset[dataset.split == "val"]["ExamDate"].value_counts())

ExamDate
2013-10-25    7
2014-08-12    5
2014-08-05    5
2015-03-23    5
2014-12-29    5
             ..
2023-08-02    1
2022-07-12    1
2022-04-20    1
2023-04-24    1
2023-08-01    1
Name: count, Length: 781, dtype: int64


ExamDate
2014-04-22    4
2014-11-20    4
2014-09-09    4
2014-06-18    4
2015-09-29    4
             ..
2022-05-04    1
2023-03-17    1
2021-11-05    1
2022-08-17    1
2023-05-31    1
Name: count, Length: 296, dtype: int64


ExamDate
2015-03-19    5
2014-12-22    4
2015-03-11    4
2015-12-11    4
2014-06-27    4
             ..
2022-02-07    1
2022-08-15    1
2021-08-25    1
2023-05-24    1
2023-08-30    1
Name: count, Length: 284, dtype: int64


In [4]:
def load_and_split(data_dir):
    paired_data = files_load(data_dir)

    train_idx, val_idx, test_idx = group_stratified_train_val_test_split(
                                                                    np.arange(len(paired_data)),
                                                                    labels=pd.to_datetime(paired_data["ExamDate"]).dt.year,
                                                                    groups=paired_data["SubjectKeyRadiology"]
                                                                        )
    paired_data = paired_data.reset_index(drop=True)

    paired_data["split"] = "unassigned"

    paired_data.loc[train_idx, "split"] = "train"
    paired_data.loc[val_idx, "split"] = "val"
    paired_data.loc[test_idx, "split"] = "test"

    return paired_data

In [ ]:

def save_final_dataset(dataset, output_root, save_root):

    # -------- output dir --------
    dataset["output_dir"] = dataset["split"].apply(
        lambda x: os.path.join(output_root, x)
    )

    # -------- clean date --------
    dates = pd.to_datetime(dataset["ExamDate"])
    dates_clean = dates.dt.date.astype(str)

    # -------- base key --------
    dataset["base_name_raw"] = (
        dataset["SubjectKeyRadiology"].astype(str)
        + "_"
        + dates_clean
    )

    # -------- count duplicates --------
    dataset["dups_count"] = dataset.groupby("base_name_raw").cumcount()

    # -------- suffix only if dup_id >= 1 --------
    def make_name(row):
        if row["dups_count"] == 0:
            return row["base_name_raw"]
        else:
            return f"{row['base_name_raw']}_{row['dup_id']}"

    dataset["base_name"] = dataset.apply(make_name, axis=1)

    # -------- output path --------
    dataset["output_path"] = dataset.apply(
        lambda row: os.path.join(row["output_dir"], f"{row['base_name']}.pt"),
        axis=1
    )

    dataset.to_csv(save_root, index=False)

    return dataset

In [13]:
data_dir = "/projects/net_contrast_classification/contrast_phase/data/cleaned_data_1.csv"
paired_data = load_and_split(data_dir)

output_root = "/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTINET/pairs_preprocessed"
save_root = "/projects/net_contrast_classification/contrast_phase/Preprocessing/Time_interval_data/paired_preprocessed_data.csv"

dataset = save_final_dataset(paired_data, output_root, save_root)